# Fitness generalization — PHASE 2: cross-ORGANISM (leave-one-organism-out)

Phase 1 showed fitness transfers across conditions within DvH (rho 0.35 vs 0.55 ceiling). Phase 2 asks the real question: does the **gene x condition -> fitness** relationship transfer to an organism the model has never seen?

**The enabling trick**: cross-organism, a gene is represented by its **orthologous group (OG)**, not its per-organism id — so an OG embedding learned on training organisms can score a held-out organism's genes. Conditions are shared by name (media / stress) across organisms.

**The risk**: joining Fitness-Browser `locusId`/`sysName` to our `orthology_features.csv` OGs. The notebook measures join coverage FIRST; if it's poor, it falls back to transferable gene features (length) only.

**Decision**: cross-organism median rho >= 0.25 = fitness generalizes across clades (major); 0.1-0.25 = weak; < 0.1 = clean negative (the residual is irreducibly experimental for new organisms).

## 1. Setup + load feba.db from Drive (no recursive glob)

In [ ]:
!pip install -q pandas numpy scipy
!git clone --depth 1 -b claude/vectorize-gex-propensity-NRqBW https://github.com/nikku03/cell.git cell_repo || echo cloned
import os; os.chdir('cell_repo')
from google.colab import drive; drive.mount('/content/drive')
# EDIT this to your actual path (from phase-1 you know where it is). No '**' globs on Drive!
SRC = '/content/drive/MyDrive/cell_count_dynamics/multiorg/fitness_browser/feba.db.gz'
assert os.path.exists(SRC), f'set SRC to your feba.db.gz path; {SRC} not found'
if not os.path.exists('/tmp/feba.db'):
    !cp "{SRC}" /tmp/feba.db.gz && time gunzip /tmp/feba.db.gz
!ls -lh /tmp/feba.db
import sqlite3, pandas as pd
con = sqlite3.connect('/tmp/feba.db')
print('tables:', pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con).name.tolist())

## 2. Build the FB-gene -> OG join (the critical step)
Map each Fitness-Browser organism to our `beril_<orgId>` name, then match FB `locusId`/`sysName` to our `orthology_features.csv` `locus_tag` to attach an OG id. Reports coverage per organism.

In [ ]:
import csv, collections, pandas as pd
# our orthology: (organism, locus_tag) -> og_id ; organism names look like beril_DvH
orth = collections.defaultdict(dict)   # beril_org -> {locus_tag: og}
for r in csv.DictReader(open('data/drive_import/labels/orthology_features.csv')):
    if r['og_id']: orth[r['organism']][r['locus_tag']] = r['og_id']
our_orgs = set(orth)
print('our orthology organisms:', len(our_orgs))

# FB organisms present
fb_orgs = pd.read_sql('SELECT DISTINCT orgId FROM GeneFitness', con).orgId.tolist()
print('FB organisms:', len(fb_orgs))
# map FB orgId -> beril_<orgId> if it exists in our orthology
def to_beril(o):
    for cand in (f'beril_{o}', o, f'beril_{o.replace("_","")}'):
        if cand in our_orgs: return cand
    # fuzzy: any our_org whose suffix matches
    for u in our_orgs:
        if u.split('_',1)[-1].lower()==o.lower(): return u
    return None
orgmap = {o: to_beril(o) for o in fb_orgs}
matched = {o:b for o,b in orgmap.items() if b}
print(f'FB orgs mapped to our orthology: {len(matched)} -> {list(matched.items())[:8]}')

# FB Gene table: locusId + sysName (try both as the join key against our locus_tag)
fbgene = pd.read_sql('SELECT orgId, locusId, sysName FROM Gene', con)
loc2og = {}   # (orgId, locusId) -> og
cov = {}
for o, b in matched.items():
    sub = fbgene[fbgene.orgId==o]; lut = orth[b]
    hit = 0
    for _, g in sub.iterrows():
        og = lut.get(str(g.locusId)) or lut.get(str(g.sysName))
        if og: loc2og[(o, g.locusId)] = og; hit += 1
    cov[o] = hit/max(1,len(sub))
print('\nper-organism OG-join coverage:')
for o in sorted(cov, key=lambda x:-cov[x])[:20]:
    print(f'  {o:<14} {cov[o]:.2%}  ({matched[o]})')
good = [o for o in cov if cov[o]>=0.4]
print(f'\norganisms with >=40% OG coverage: {len(good)} -> {good}')

## 3. Build the cross-organism tensor (OG embedding + global condition vocab)
Restrict to organisms with good OG coverage AND >=40 conditions. Conditions shared by name across organisms (global vocab). Fitness z-normalized per experiment.

In [ ]:
import numpy as np, pandas as pd
exp = pd.read_sql('SELECT orgId, expName, media, aerobic, condition_1 FROM Experiment', con)
ncond = exp.groupby('orgId').size()
USE = [o for o in good if ncond.get(o,0)>=40]
print(f'organisms used (good OG + >=40 conditions): {len(USE)} -> {USE}')
assert len(USE)>=4, 'need >=4 organisms for a leave-one-out test; relax thresholds if needed'

# global vocabularies (shared across organisms)
frames=[]
for o in USE:
    gf = pd.read_sql(f"SELECT orgId, locusId, expName, fit FROM GeneFitness WHERE orgId='{o}'", con)
    gf['og'] = [loc2og.get((o, l)) for l in gf.locusId]
    gf = gf[gf.og.notna()]
    gf = gf.merge(exp[exp.orgId==o][['expName','media','condition_1','aerobic']], on='expName', how='inner')
    gf['fit_z'] = gf.groupby('expName').fit.transform(lambda x:(x-x.median())/(x.std()+1e-6))
    frames.append(gf)
D = pd.concat(frames, ignore_index=True)
print('cross-org training pairs:', len(D), '| organisms:', D.orgId.nunique(), '| OGs:', D.og.nunique())
# integer codes
D['og_i']   = D.og.astype('category').cat.codes
D['med_i']  = D.media.fillna('?').astype('category').cat.codes
D['cond_i'] = D.condition_1.fillna('none').astype('category').cat.codes
n_og=int(D.og_i.max())+1; n_med=int(D.med_i.max())+1; n_cond=int(D.cond_i.max())+1
print(f'n_og={n_og} n_med={n_med} n_cond={n_cond}')

## 4. Leave-one-ORGANISM-out training
Embed OG (shared across organisms), media, condition. Hold out one organism entirely; its genes' OGs (mostly seen in training) carry the learned embedding. Report per-organism Spearman + each org's noise floor.

In [ ]:
import numpy as np
from scipy.stats import spearmanr
E_OG=32; E_C=16; HID=64
def init(rng):
    return dict(eog=rng.normal(0,.3,(n_og,E_OG)), em=rng.normal(0,.3,(n_med,E_C)),
                ec=rng.normal(0,.3,(n_cond,E_C)),
                W1=rng.normal(0,.3,(E_OG+2*E_C,HID)), b1=np.zeros(HID),
                w2=rng.normal(0,.3,HID), b2=0.0)
def fwd(P, oi, mi, ci):
    x=np.concatenate([P['eog'][oi],P['em'][mi],P['ec'][ci]],1)
    h=np.maximum(x@P['W1']+P['b1'],0); return (h@P['w2']+P['b2']), x, h

def noise_floor(o):
    ex=exp[exp.orgId==o].copy(); ex['cl']=ex.media.fillna('-')+'|'+ex.condition_1.fillna('-')
    gf=pd.read_sql(f"SELECT locusId, expName, fit FROM GeneFitness WHERE orgId='{o}'", con)
    rs=[]
    for cl,sub in ex.groupby('cl'):
        n=sub.expName.tolist()
        for i in range(len(n)):
            for j in range(i+1,min(i+2,len(n))):
                a=gf[gf.expName==n[i]].set_index('locusId').fit; b=gf[gf.expName==n[j]].set_index('locusId').fit
                c=a.index.intersection(b.index)
                if len(c)>200: rs.append(spearmanr(a.loc[c],b.loc[c]).statistic)
    rs=[r for r in rs if r==r]; return float(np.median(rs)) if rs else float('nan')

results=[]
for held in USE:
    tr=D[D.orgId!=held]; te=D[D.orgId==held]
    if len(te)<500: continue
    rng=np.random.default_rng(0); P=init(rng); M={k:0 for k in P}; V=dict(M); t=0; lr=3e-3
    oi=tr.og_i.to_numpy(); mi=tr.med_i.to_numpy(); ci=tr.cond_i.to_numpy(); yt=tr.fit_z.to_numpy(); n=len(tr); bs=4096
    for ep in range(4):
        idx=rng.permutation(n)
        for s in range(0,n,bs):
            b=idx[s:s+bs]; yh,x,h=fwd(P,oi[b],mi[b],ci[b]); err=(yh-yt[b])/len(b)
            gw2=h.T@err; gb2=err.sum(); dh=np.outer(err,P['w2'])*(h>0)
            gW1=x.T@dh; gb1=dh.sum(0); dx=dh@P['W1'].T
            d_og=dx[:,:E_OG]; d_m=dx[:,E_OG:E_OG+E_C]; d_c=dx[:,E_OG+E_C:E_OG+2*E_C]
            grads={'W1':gW1,'b1':gb1,'w2':gw2,'b2':gb2}
            for emb,ii,d in [('eog',oi[b],d_og),('em',mi[b],d_m),('ec',ci[b],d_c)]:
                g=np.zeros_like(P[emb]); np.add.at(g,ii,d); grads[emb]=g
            t+=1
            for k in grads:
                if not isinstance(M[k],np.ndarray): M[k]=np.zeros_like(grads[k]); V[k]=np.zeros_like(grads[k])
                M[k]=0.9*M[k]+0.1*grads[k]; V[k]=0.999*V[k]+0.001*(grads[k]**2)
                P[k]-=lr*(M[k]/(1-0.9**t))/(np.sqrt(V[k]/(1-0.999**t))+1e-8)
    yh,_,_=fwd(P, te.og_i.to_numpy(), te.med_i.to_numpy(), te.cond_i.to_numpy())
    rho=spearmanr(yh, te.fit_z.to_numpy()).statistic
    nf=noise_floor(held)
    results.append((held, len(te), rho, nf))
    print(f'  held={held:<14} n_test={len(te):>7}  cross-org rho={rho:.3f}  (noise floor {nf:.3f})')
import numpy as np
med=np.median([r for _,_,r,_ in results])
print(f'\nCROSS-ORGANISM median rho = {med:.3f}')

## 5. Interpret + save

In [ ]:
import json, numpy as np
med=float(np.median([r for _,_,r,_ in results]))
verdict=('GENERALIZES across clades (major)' if med>=0.25 else
         'weak cross-organism transfer' if med>=0.10 else
         'NO cross-organism transfer (residual is irreducibly experimental)')
out={'phase':'cross_organism_LOO','n_orgs':len(results),
     'cross_org_median_rho':med,'verdict':verdict,
     'per_org':[{'org':o,'n_test':int(n),'cross_org_rho':float(r),'noise_floor':float(nf)} for o,n,r,nf in results]}
json.dump(out, open('outputs/orphan/fitness_generalize_cross_org.json','w'), indent=2)
import shutil; shutil.copy('outputs/orphan/fitness_generalize_cross_org.json','/content/drive/MyDrive/fitness_generalize_cross_org.json')
print(json.dumps(out, indent=2))
print('\nVERDICT:', verdict)